# 基线实验与模型对比

运行完整推荐 pipeline，对比 6 种模型的评估指标，并进行消融实验、用户分群分析和流行度偏差诊断。

In [ ]:
from __future__ import annotations
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

from src.data.feature_engineering import build_feature_store
from src.data.preprocess import clean_movies, clean_ratings, clean_tags, leave_one_out_split
from src.data.load_data import make_sample_movielens
from src.models import build_default_models
from src.evaluation.evaluate import evaluate_models
from src.evaluation.ablation import run_ablation_experiments
from src.evaluation.segments import evaluate_user_segments
from src.evaluation.diagnostics import popularity_bias_report
from src.visualization.plot_results import plot_metric_comparison
from src.models.hybrid import HybridRecommender
from src.visualization.explain_recommendation import explain_recommendations

%matplotlib inline

## 1. 数据准备

In [ ]:
USE_SAMPLE = True
K = 5

if USE_SAMPLE:
    data = make_sample_movielens()
else:
    from src.data.load_data import load_movielens, resolve_movielens_dir
    data_dir = resolve_movielens_dir()
    data = load_movielens(data_dir, max_ratings=50000, max_tags=5000)

movies = clean_movies(data["movies"])
ratings = clean_ratings(data["ratings"], min_user_ratings=1, min_movie_ratings=1)
tags = clean_tags(data["tags"])

movies = movies[movies["movieId"].isin(ratings["movieId"])]

train_ratings, test_ratings = leave_one_out_split(ratings, min_interactions=3, relevant_threshold=4.0)

features = build_feature_store(train_ratings, movies, tags)

print(f"训练集: {len(train_ratings):,} 条评分")
print(f"测试集: {len(test_ratings):,} 条评分")
print(f"用户数: {features.ratings['userId'].nunique():,}")
print(f"电影数: {features.movies['movieId'].nunique():,}")
print(f"内容特征维度: {features.content_features.shape[1]}")

## 2. 训练所有模型

In [ ]:
models = build_default_models()
for name, model in models.items():
    model.fit(features)
    print(f"  {name}: fitted")
print(f"\n已训练 {len(models)} 个模型")

## 3. 模型评估对比

In [ ]:
evaluation = evaluate_models(models, features, test_ratings, k=K)
evaluation

In [ ]:
metric_cols = ["precision_at_k", "recall_at_k", "ndcg_at_k", "hit_rate_at_k", "map_at_k", "mrr_at_k", "coverage", "diversity"]

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, metric in enumerate(metric_cols):
    ax = axes[i]
    sorted_df = evaluation.sort_values(metric, ascending=False)
    colors = ["steelblue" if m != "hybrid" else "coral" for m in sorted_df["model"]]
    ax.barh(sorted_df["model"], sorted_df[metric], color=colors)
    ax.set_title(metric)
    ax.set_xlabel("score")

plt.suptitle("Model Comparison Across All Metrics", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. 消融实验

In [ ]:
ablation = run_ablation_experiments(features, test_ratings, k=K, max_users=100)
ablation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ablation_sorted = ablation.sort_values("ndcg_at_k", ascending=True)
ax.barh(ablation_sorted["ablation_setting"], ablation_sorted["ndcg_at_k"], color="steelblue")
ax.set_title("Ablation Study: NDCG@K by Component")
ax.set_xlabel("NDCG@K")
plt.tight_layout()
plt.show()

## 5. 用户活跃度分群分析

In [ ]:
segments = evaluate_user_segments(models, features, test_ratings, k=K, max_users_per_segment=100)
segments

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (segment_name, group) in enumerate(segments.groupby("activity_segment")):
    ax = axes[i]
    group_sorted = group.sort_values("ndcg_at_k", ascending=False)
    ax.barh(group_sorted["model"], group_sorted["ndcg_at_k"], color="steelblue")
    ax.set_title(f"{segment_name}")
    ax.set_xlabel("NDCG@K")
plt.suptitle("NDCG@K by User Activity Segment", fontsize=14)
plt.tight_layout()
plt.show()

## 6. 流行度偏差分析

In [ ]:
bias = popularity_bias_report(models, features, k=K, max_users=100, top_quantile=0.80)
bias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bias_sorted = bias.sort_values("popular_item_ratio", ascending=True)
ax1 = axes[0]
ax1.barh(bias_sorted["model"], bias_sorted["popular_item_ratio"], color="steelblue")
ax1.set_title("Popular Item Ratio (lower = less bias)")
ax1.set_xlabel("Ratio")

ax2 = axes[1]
ax2.barh(bias_sorted["model"], bias_sorted["avg_weighted_rating"], color="coral")
ax2.set_title("Avg Weighted Rating of Recommendations")
ax2.set_xlabel("Avg Weighted Rating")

plt.tight_layout()
plt.show()

## 7. 示例推荐 + 解释

In [ ]:
hybrid = models["hybrid"]
sample_user = int(features.ratings["userId"].iloc[0])
print(f"用户 {sample_user} 的高分电影:")
user_data = features.ratings[features.ratings["userId"] == sample_user].nlargest(5, "rating")
for _, row in user_data.iterrows():
    title = features.movies.loc[features.movies["movieId"] == row["movieId"], "title"].values[0]
    print(f"  {title} (rating={row['rating']})")

print(f"\n混合模型 Top-{K} 推荐:")
recs = hybrid.recommend(sample_user, k=K, exclude_seen=True)
recs_df = explain_recommendations(hybrid, sample_user, features, k=K)
for _, row in recs_df.iterrows():
    print(f"  {row['title']} | score={row['score']:.4f} | {row['reason']}")

## 8. 冷启动推荐演示

In [ ]:
from src.models.cold_start import recommend_for_new_user

cold_recs = recommend_for_new_user(
    features,
    genres=["Action", "Sci-Fi"],
    keywords="space adventure robot",
    k=K,
)
print("冷启动推荐 (Action + Sci-Fi, keywords='space adventure robot'):")
cold_recs[["title", "score", "reason"]]

## 9. 保存结果

In [ ]:
from src.config import REPORTS_DIR
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

evaluation.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
ablation.to_csv(REPORTS_DIR / "ablation_results.csv", index=False)
segments.to_csv(REPORTS_DIR / "user_segment_results.csv", index=False)
bias.to_csv(REPORTS_DIR / "popularity_bias.csv", index=False)

print(f"结果已保存至 {REPORTS_DIR}")